# **Implementación del Algoritmo de Shor para resolver el caso general del DLP**

## Comparativa entre los oráculos para $p=23$

### Rodrigo Hernández Sacristán

#### Máster en Computación Cuántica, Universidad Internacional de la Rioja (UNIR)

Este archivo es un script auxiliar de análisis (NO es un programa de resolución del DLP como tal).
Genera las cifras de la tabla comparativa de los tres oráculo (Tabla 8.1 de la memoria): número de cúbits, puertas CX y profundidad.
 
Metodología (importante para la reproducibilidad):
  - Para cada oráculo se monta el circuito COMPLETO del DLP a p=23: superposición (H) -> oráculo -> QFT -> medida.
  - Se transpila a la base de puertas elementales {cx, rz, sx, x} con optimization_level=1 y SIN backend, es decir, asumiendo CONECTIVIDAD COMPLETA (no se insertan SWAPs).
  - De ahí se leen las puertas CX (count_ops) y la profundidad (depth).
 
Estas cifras son comparativas entre oráculo luego mo coinciden con las del mapeo a un procesador real (ibm_marrakesh usa 'cz' como puerta nativa y
tiene conectividad heavy-hex, que introduce SWAPs adicionales y dispara tanto el número de puertas como la profundidad).

In [1]:
## LIBRERÍAS NECESARIAS 

import numpy as np
import math
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate 
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator
from qiskit import transpile
from qiskit.visualization import plot_histogram
from IPython.display import display
from qiskit.quantum_info import Operator
from qiskit.circuit.library import QFT
import warnings
import random

## FUNCIONES EXTERNAS NECESARIAS

In [2]:
def es_generador(g, p):
    """Verifica si g es generador de Z_p^*"""
    orden = p - 1
    elementos_vistos = set()
    for k in range(1, orden + 1):
        elementos_vistos.add(pow(g, k, p))
    return len(elementos_vistos) == orden

def residuo_centrado(z, q):
    r = z % q
    if r > q // 2:
        r -= q
    return r

def extraer_modulo_coprimo(c_prima, mod_total):
    """ Extrae el mayor divisor 'm' de mod_total tal que gcd(c_prima, m) == 1 """
    m = mod_total
    while True:
        g = math.gcd(c_prima, m)
        if g == 1:
            break
        m = m // g
    return m

def combinar_teorema_chino(X, M, a, m):
    """ Combina x ≡ X (mod M) y x ≡ a (mod m) usando el Teorema Chino del Resto """
    g = math.gcd(M, m)
    if (a - X) % g != 0:
        return None, None  # Inconsistencia por ruido de fase
    
    m_red = m // g
    M_red = M // g
    inv = pow(M_red, -1, m_red)
    diff = (a - X) // g
    k = (diff * inv) % m_red
    
    M_nuevo = (M * m) // g
    X_nuevo = (X + k * M) % M_nuevo
    return X_nuevo, M_nuevo



## ORÁCULO BASADO EN MATRICES $O(4^{l})$

In [ ]:
def aplicar_oraculo_matrices(g, h, p, qc, reg_A, reg_B, reg_C, reg_anc=None):
    '''
    Implementa el oráculo cuántico para f(a,b) = g^a * h^(-b) mod p usando matrices de permutación
    '''
    
    num_qubits_reg = len(reg_A)
    q= 2**num_qubits_reg  # si  l=5 cúbits pues q=32
    
    # ----------------------------------------------------------------------
    # PASO 1: Inicializar el Registro C en |1> (ya que empezamos multiplicando)
    # ----------------------------------------------------------------------
    # El registro C por defecto está en |00000>, aplicamos una puerta X (NOT)
    # en el primer cúbit para obtener el estado clásico |1> (00001 en binario)
    qc.x(reg_C[0])
    
    # ----------------------------------------------------------------------
    # PASO 2: Calcular las potencias clásicas de las bases mod p
    # ----------------------------------------------------------------------
    # Registro A:
    factores_A = [pow(g, 2**i, p) for i in range(num_qubits_reg)]
    
    # Registro B: potencias del inverso de h. 
    h_inv = pow(h, -1, p)
    factores_B = [pow(h_inv, 2**j, p) for j in range(num_qubits_reg)]
    

    # -------------------------------------------------------------------------------------------------
    # PASO 3: Función interna para crear la matriz de multiplicación modular
    # -------------------------------------------------------------------------------------------------
    def crear_matriz_multiplicacion(k): 
        matriz = np.zeros((q, q)) # Matriz qxq rellena de 0s
        
        for x in range(q):
            if x < p: # Para estados válidos dentro de Z_p^*
                if x == 0:
                    estado_destino = 0
                else:
                    estado_destino = (x * k) % p
            else: 
                # Para  los estados que están fuera del grupo, los mapeamos a sí mismos
                # para mantener la matriz unitaria (identidad en la esquina inferior derecha)
                estado_destino = x
                
            matriz[estado_destino, x] = 1 # Matriz de permutación unitaria
            
        return UnitaryGate(matriz, label=f"x*{k} mod {p}")

    # ----------------------------------------------------------------------
    # PASO 4: Aplicar multiplicaciones controladas por el Registro A
    # ----------------------------------------------------------------------
    for i in range(num_qubits_reg):
        potencia = factores_A[i]
        
        # Optimización: si multiplicar por 1 no cambia el estado, nos saltamos la puerta
        if potencia == 1:
            continue
            
        puerta_mult = crear_matriz_multiplicacion(potencia)
        puerta_controlada = puerta_mult.control(1)
        
        # El cúbit de control es reg_A[i], y actúa sobre todo el reg_C
        qc.append(puerta_controlada, [reg_A[i]] + list(reg_C)) 
        
    # ----------------------------------------------------------------------
    # PASO 5: Aplicar multiplicaciones controladas por el Registro B
    # ----------------------------------------------------------------------
    for j in range(num_qubits_reg):
        potencia = factores_B[j]
        
        if potencia == 1:
            continue
            
        puerta_mult = crear_matriz_multiplicacion(potencia)
        puerta_controlada = puerta_mult.control(1)
        
        # El cúbit de control es reg_B[j], y actúa sobre todo el reg_C
        qc.append(puerta_controlada, [reg_B[j]] + list(reg_C))

## ORÁCULO BASADO EN PUERTAS: BEAUREGARD $O(l^4)$

In [ ]:
"""
Oráculo cuántico para el algoritmo de Shor (logaritmo discreto) que
implementa la EXPONENCIACIÓN MODULAR mediante puertas cuánticas
(sumadores de Draper en espacio de Fourier + esquema de Beauregard),
en sustitución de la versión basada en matrices de permutación 2^l x 2^l.

    |a>|b>|0...0>  -->  |a>|b>| g^a * h^(-b) mod p >

El registro de valor (reg_C) se multiplica EN SITIO, primero por las
potencias de g (controladas por reg_A) y después por las potencias de
h^(-1) (controladas por reg_B). Las ancillas empiezan y terminan en |0>.

Coste de qubits del registro inferior (para módulo p):
    size      = ceil(log2(p))          # qubits para el valor  
    n_ancilla = size + 2               # ancillas auxiliares    
Por ejemplo, para p = 23:  size = 5,  n_ancilla = 7.

==================================================================
"""


# ------------------------------------------------------------------
# QFT sin swaps 
# ------------------------------------------------------------------
def _qft_ns_brg(n):
    qc = QuantumCircuit(n, name="QFT")
    for j in range(n - 1, -1, -1):
        qc.h(j)
        for k in range(j - 1, -1, -1):
            qc.cp(math.pi / 2 ** (j - k), k, j)
    return qc


def _angle_brg(k, b):
    """Ángulo de rotación de Draper: b*pi / 2^k."""
    return b * np.pi / (2 ** k)


# ------------------------------------------------------------------
# Sumadores en espacio de Fourier 
# ------------------------------------------------------------------
def _phi_add_brg(n, b, factor):
    """|phi(x)> -> |phi(x + factor*b)>  (sin control)."""
    qc = QuantumCircuit(n, name=f"add({b})")
    for k in range(n):
        qc.p(factor * _angle_brg(k, b), k)
    return qc


def _phi_add_c_brg(n, b, factor):
    """Sumador con 1 control."""
    ctrl = QuantumRegister(1, "c")
    reg = QuantumRegister(n, "r")
    qc = QuantumCircuit(ctrl, reg, name=f"cadd({b})")
    for k in range(n):
        qc.cp(factor * _angle_brg(k, b), ctrl[0], reg[k])
    return qc


def _phi_add_cc_brg(n, b, factor):
    """Sumador con 2 controles."""
    ctrl = QuantumRegister(2, "c")
    reg = QuantumRegister(n, "r")
    qc = QuantumCircuit(ctrl, reg, name=f"ccadd({b})")
    for k in range(n):
        qc.mcp(factor * _angle_brg(k, b), [ctrl[0], ctrl[1]], reg[k])
    return qc


# ------------------------------------------------------------------
# Sumador modular doble-controlado  (Beauregard)
#   |phi(a)>|0>_aux -> |phi((a+y) mod N)>|0>_aux
#   El registro 'reg' entra y sale en espacio de Fourier.
# ------------------------------------------------------------------
def _phi_add_mod_N_cc_brg(n, y, N):
    ctrl = QuantumRegister(2, "ctrl")
    reg = QuantumRegister(n, "a")
    aux = QuantumRegister(1, "aux")
    qc = QuantumCircuit(ctrl, reg, aux, name=f"ccADD({y})MOD({N})")

    iqft = _qft_ns_brg(n).inverse()
    qft = _qft_ns_brg(n)

    qc.append(_phi_add_cc_brg(n, y, 1).to_gate(), list(ctrl) + list(reg))
    qc.append(_phi_add_brg(n, N, -1).to_gate(), reg)
    qc.append(iqft.to_gate(), reg)
    qc.cx(reg[n - 1], aux[0])
    qc.append(qft.to_gate(), reg)
    qc.append(_phi_add_c_brg(n, N, 1).to_gate(), [aux[0]] + list(reg))
    qc.append(_phi_add_cc_brg(n, y, -1).to_gate(), list(ctrl) + list(reg))
    qc.append(iqft.to_gate(), reg)
    qc.x(reg[n - 1])
    qc.cx(reg[n - 1], aux[0])
    qc.x(reg[n - 1])
    qc.append(qft.to_gate(), reg)
    qc.append(_phi_add_cc_brg(n, y, 1).to_gate(), list(ctrl) + list(reg))
    return qc


# ------------------------------------------------------------------
# Multiplicación modular controlada (en sitio)
#   |c>|x>|0>|0> -> |c>| (x * a mod N) si c=1, si no x >|0>|0>
# ------------------------------------------------------------------
def _mult_mod_N_partial_c_brg(y, N, size_x, size_b):
    """Multiplicación-suma fuera de sitio: |c>|x>|b>|anc> -> |c>|x>|b + y*x mod N>|anc>."""
    ctrl = QuantumRegister(1, "c")
    rx = QuantumRegister(size_x, "x")
    rb = QuantumRegister(size_b, "b")
    anc = QuantumRegister(1, "anc")
    qc = QuantumCircuit(ctrl, rx, rb, anc, name=f"CMUL0({y})")

    qc.append(_qft_ns_brg(size_b).to_gate(), rb)
    for k in range(size_x):
        add = _phi_add_mod_N_cc_brg(size_b, (2 ** k * y) % N, N)
        qc.append(add.to_gate(), [ctrl[0], rx[k]] + list(rb) + [anc[0]])
    qc.append(_qft_ns_brg(size_b).inverse().to_gate(), rb)
    return qc


def _mult_mod_N_c_brg(a, N):
    """Multiplicación modular controlada EN SITIO: |c>|x>|0>|0> -> |c>| a*x mod N >|0>|0>.
    Requiere gcd(a, N) = 1 (siempre cierto aquí porque N = p es primo)."""
    size = math.ceil(math.log2(N))
    size_x = size
    size_b = size + 1

    ctrl = QuantumRegister(1, "c")
    rx = QuantumRegister(size_x, "x")
    rb = QuantumRegister(size_b, "b")
    anc = QuantumRegister(1, "anc")
    qc = QuantumCircuit(ctrl, rx, rb, anc, name=f"CMUL({a})MOD({N})")

    qc.append(_mult_mod_N_partial_c_brg(a, N, size_x, size_b).to_gate(), qc.qubits)
    for i in range(size_x):
        qc.cswap(ctrl[0], rx[i], rb[i])
    inv_a = pow(a, -1, N)
    qc.append(_mult_mod_N_partial_c_brg(inv_a, N, size_x, size_b).inverse().to_gate(), qc.qubits)
    return qc


# ------------------------------------------------------------------
# Exponenciación modular:  |x>|input> -> |x>| (valor * a^x) mod N >
#   El registro 'input' tiene 2*ceil(log2 N)+2 qubits:
#     [ valor (size) | b (size+1) | anc (1) ]
#   El valor vive en los primeros 'size' qubits de 'input'.
# ------------------------------------------------------------------
def _mod_exp_brg(n, a, N):
    size = math.ceil(math.log2(N))
    size_input = 2 * size + 2
    xr = QuantumRegister(n, "x")
    inp = QuantumRegister(size_input, "input")
    qc = QuantumCircuit(xr, inp, name=f"{a}^x mod {N}")
    for i in range(n):
        cte = pow(a, 2 ** i, N)   # constante a^(2^i) mod N para este bit del exponente
        # multiplicar por 1 es la identidad, así que nos saltamos por completo esa multiplicación controlada.
        if cte == 1:
            continue
        g = _mult_mod_N_c_brg(cte, N)
        qc.append(g.to_gate(), [xr[i]] + list(inp))
    return qc


def num_ancillas_beauregard(p):
    """Número de qubits ancilla que necesita el oráculo para el módulo p."""
    return math.ceil(math.log2(p)) + 2


def aplicar_oraculo_beauregard(g, h, p, qc, reg_A, reg_B, reg_C, reg_anc):
    """
    Implementa f(a,b) = g^a * h^(-b) mod p mediante puertas cuánticas.

        |a>|b>|1>|0...0>  -->  |a>|b>| g^a * h^(-b) mod p >|0...0>

    Parámetros
    ----------
    reg_A, reg_B : registros de exponente (l qubits cada uno).
    reg_C        : registro de VALOR (l qubits). Debe cumplir l >= ceil(log2 p).
    reg_anc      : registro ancilla NUEVO. Tamaño = num_ancillas_beauregard(p)
                   (para p=23 -> 7 qubits). Entra y sale en |0>.
    """
    l = len(reg_A)
    size = math.ceil(math.log2(p))
    assert len(reg_C) >= size, f"reg_C necesita >= {size} qubits para p={p}"
    assert len(reg_anc) == num_ancillas_beauregard(p), \
        f"reg_anc debe tener {num_ancillas_beauregard(p)} qubits para p={p}, tiene {len(reg_anc)}"

    # El 'input' que espera _mod_exp es [valor(size) | b(size+1) | anc(1)].
    # Aquí: valor = reg_C, y el resto (size+1 + 1 = size+2) son las ancillas.
    bottom = list(reg_C) + list(reg_anc)

    # Inicializamos el registro de valor a |1> (empezamos a multiplicar desde 1)
    qc.x(reg_C[0])

    h_inv = pow(h, -1, p)
    print(f"[Cuántico] Exponenciación por puertas: base g={g} (reg_A), base h^-1={h_inv} (reg_B)")

    # 1) Multiplica en sitio por g^a  (controlado por reg_A)
    qc.append(_mod_exp_brg(l, g, p).to_gate(label=f"g^a mod {p}"), list(reg_A) + bottom)

    # 2) Multiplica en sitio por (h^-1)^b = h^-b  (controlado por reg_B)
    qc.append(_mod_exp_brg(l, h_inv, p).to_gate(label=f"h^-b mod {p}"), list(reg_B) + bottom)


## ORÁCULO BASADO EN PUERTAS (MONTGOMERY) $O(l^3)$

In [ ]:
"""
Oráculo cuántico para Shor-DLP con MULTIPLICACIÓN MODULAR DE MONTGOMERY

    - Exponenciación modular en  O(n^3)   
    - Número de ancillas: 2n+1     

    |a>|b>||0..0>  ->  |a>|b>| g^a * h^(-b) mod p >

Recuento de qubits (para módulo p, con n = ceil(log2 p)):
    reg_C (valor) = n           <- tu registro de valor (smallreg)
    reg_anc       = 2n + 1       <- espacio de trabajo Montgomery
Por ejemplo para p=23 (n=5): reg_anc = 11 qubits, total del oráculo = 3n + (2n+1) = 26.

La clave de Montgomery: la reducción modular extrae el bit MENOS significativo
con una sola Hadamard (en vez del bit MÁS significativo con una QFT completa
en cada suma, como en Beauregard). Eso es lo que baja el coste a O(n^3).

"""

warnings.filterwarnings("ignore", category=DeprecationWarning)


def _qft(m):
    return QFT(m, do_swaps=False).to_gate()


def _iqft(m):
    return QFT(m, do_swaps=False).inverse().to_gate()


def _angle(k, b):
    """Ángulo de Draper: b*pi / 2^k."""
    return b * np.pi / (2 ** k)


# ------------------------------------------------------------------
# Sumadores de Draper en espacio de Fourier
# ------------------------------------------------------------------
def _ccp(theta):
    """Puerta de fase doble-controlada mediante fases simples (medio ángulo)."""
    ctrl = QuantumRegister(2, "ctrl")
    q = QuantumRegister(1, "q")
    c = QuantumCircuit(ctrl, q, name="CCP")
    c.cp(theta / 2, ctrl[1], q)
    c.cx(ctrl[0], ctrl[1])
    c.cp(-theta / 2, ctrl[1], q)
    c.cx(ctrl[0], ctrl[1])
    c.cp(theta / 2, ctrl[0], q)
    return c.to_gate()


def _phi_add_cc(n, b, factor):
    """Suma doble-controlada: |phi(x)> -> |phi(x + factor*b)>."""
    ctrl = QuantumRegister(2, "c")
    reg = QuantumRegister(n, "r")
    a = QuantumCircuit(ctrl, reg, name="CCADD")
    for k in range(n):
        a.append(_ccp(factor * _angle(k, b)), list(ctrl) + [reg[k]])
    return a.to_gate()


def _phi_add_c_ignore(n, b, factor, ignorebits):
    """Suma controlada que IGNORA los 'ignorebits' bits menos significativos
    (los ya fijados en |u> durante la reducción de Montgomery)."""
    ctrl = QuantumRegister(1, "c")
    reg = QuantumRegister(n - ignorebits, "r")
    a = QuantumCircuit(ctrl, reg, name="CADDign")
    for k in range(n):
        if k < ignorebits:
            continue
        a.cp(factor * _angle(k, b), ctrl[0], reg[k - ignorebits])
    return a.to_gate()


def _phi_add_c(n, b, factor):
    return _phi_add_c_ignore(n, b, factor, 0)


# ------------------------------------------------------------------
# Multiplicación modular de Montgomery
# ------------------------------------------------------------------
def _mult_montgomery_partial_c(n, y_montg, p):
    """Multiplicación fuera de sitio + reducción de Montgomery:
    |x>|0..0> -> |x>|0>|xy mod p>|0..0>.  'y_montg' entra en forma de Montgomery (yR mod p)."""
    ctrl = QuantumRegister(1, "ctrl")
    small = QuantumRegister(n, "small")
    big = QuantumRegister(2 * n + 1, "big")
    m = QuantumCircuit(ctrl, small, big, name="MMULp")

    # (1) multiplicación por sumas repetidas en espacio de Fourier
    m.append(_qft(2 * n + 1), big)
    for i in range(n):
        s = (2 ** i * y_montg) % p
        m.append(_phi_add_cc(2 * n + 1, s, 1), [ctrl[0], small[i]] + list(big))

    # (2) reducción de Montgomery: en cada iteración se extrae el LSB con una
    #     Hadamard y se usa para controlar una resta de p; la división por 2 es
    #     implícita porque se trabaja sobre un registro cada vez más pequeño.
    for i in range(n):
        qubits = [big[i]] + list(big[i + 1:])
        m.h(big[i])
        m.append(_phi_add_c_ignore(2 * n + 1 - i, int(p), -1, 1), qubits)

    # (3) corrección de signo: extraer el bit de signo y sumar p si es negativo
    m.append(_iqft(n + 1), big[n:])
    signbit = big[-1]
    m.append(_qft(n), big[n:2 * n])
    m.append(_phi_add_c(n, p, 1), [signbit] + list(big[n:2 * n]))
    m.h(big[n]); m.cx(big[n], signbit); m.h(big[n])

    # (4) descomputación del registro u (restas con p^{-1} mod 2^{n+1})
    uncom = list(big[:n]) + [big[-1]]
    m.append(_qft(n + 1), uncom)
    pinv = pow(p, -1, 2 ** (n + 1))
    for i in range(n):
        s = ((2 ** i * y_montg % p) * pinv) % 2 ** (n + 1)
        m.append(_phi_add_cc(n + 1, s, -1), [ctrl[0], small[i]] + uncom)

    m.append(_iqft(n), big[n:2 * n])
    m.append(_iqft(n + 1), uncom)
    return m.to_gate()


def _mult_montgomery_c(n, y, p):
    """Multiplicación modular controlada EN SITIO: |c>|x>|0> -> |c>| x*y mod p >|0>.
    Dos multiplicaciones fuera de sitio (con y y con y^{-1}) más un SWAP."""
    ctrl = QuantumRegister(1, "ctrl")
    small = QuantumRegister(n, "small")
    big = QuantumRegister(2 * n + 1, "big")
    m = QuantumCircuit(ctrl, small, big, name="MMUL")

    R = 2 ** n
    y_montg = y * R % p
    m.append(_mult_montgomery_partial_c(n, y_montg, p), m.qubits)
    for i in range(n):
        m.cswap(ctrl[0], small[i], big[n + i])
    iy_montg = pow(y, -1, p) * R % p
    m.append(_mult_montgomery_partial_c(n, iy_montg, p).inverse(), m.qubits)
    return m.to_gate()


def _mod_exp_montgomery(n, a, p):
    """|x>|input> -> |x>| valor*a^x mod p >.  input = 3n+1 qubits: [valor(n) | big(2n+1)]."""
    top = QuantumRegister(n, "top")
    bot = QuantumRegister(3 * n + 1, "bot")
    c = QuantumCircuit(top, bot, name=f"{a}^x mod {p}")
    for i in range(n):
        cte = pow(a, 2 ** i, p)
        if cte == 1:          # multiplicar por 1 es la identidad -> se salta
            continue
        c.append(_mult_montgomery_c(n, cte, p), [top[i]] + list(bot))
    return c.to_gate()



def num_ancillas_montgomery(p):
    """Ancillas que necesita el oráculo Montgomery: 2n+1 (para p=23 -> 11)."""
    return 2 * math.ceil(math.log2(p)) + 1


def aplicar_oraculo_montgomery(g, h, p, qc, reg_A, reg_B, reg_C, reg_anc):
    """
    Implementa f(a,b) = g^a * h^(-b) mod p con multiplicación de Montgomery.
    """
    n = len(reg_A)
    size = math.ceil(math.log2(p))
    assert len(reg_C) >= size, f"reg_C necesita >= {size} qubits para p={p}"
    assert len(reg_anc) == num_ancillas_montgomery(p), \
        f"reg_anc debe tener {num_ancillas_montgomery(p)} qubits para p={p}, tiene {len(reg_anc)}"

    bottom = list(reg_C) + list(reg_anc)   # smallreg(n) + big(2n+1)
    qc.x(reg_C[0])                         # valor inicial |1>
    h_inv = pow(h, -1, p)
    qc.append(_mod_exp_montgomery(n, g, p),     list(reg_A) + bottom)
    qc.append(_mod_exp_montgomery(n, h_inv, p), list(reg_B) + bottom)

## Programa que muestra la comparativa de los oráculos

In [8]:
g, h, p, l = 5, 19, 23, 5        # p=23  ->  n = l = 5
BASIS = ['cx', 'rz', 'sx', 'x']  # base de puertas elementales
OPT = 1                          # optimization_level
 
ORACULOS = [
    ("Matrices",   aplicar_oraculo_matrices,   lambda p: 0),
    ("Beauregard", aplicar_oraculo_beauregard, num_ancillas_beauregard),
    ("Montgomery", aplicar_oraculo_montgomery, num_ancillas_montgomery),
]
 
 
def construir_circuito(aplicar_oraculo, num_ancillas):
    """Monta el circuito completo del DLP a p=23 para un oráculo dado."""
    reg_A = QuantumRegister(l, 'A')
    reg_B = QuantumRegister(l, 'B')
    reg_C = QuantumRegister(l, 'C')
    cl_A = ClassicalRegister(l, 'medida_c')
    cl_B = ClassicalRegister(l, 'medida_d')
 
    n_anc = num_ancillas(p)
    if n_anc > 0:
        reg_anc = QuantumRegister(n_anc, 'anc')
        qc = QuantumCircuit(reg_A, reg_B, reg_C, reg_anc, cl_A, cl_B)
    else:
        reg_anc = None
        qc = QuantumCircuit(reg_A, reg_B, reg_C, cl_A, cl_B)
 
    qc.h(reg_A)
    qc.h(reg_B)
    aplicar_oraculo(g, h, p, qc, reg_A, reg_B, reg_C, reg_anc)
    qc.append(QFTGate(num_qubits=l), reg_A)
    qc.append(QFTGate(num_qubits=l), reg_B)
    qc.measure(reg_A, cl_A)
    qc.measure(reg_B, cl_B)
    return qc
 
 
def medir(aplicar_oraculo, num_ancillas):
    """Devuelve (qubits, CX, profundidad, total_puertas) tras transpilar."""
    qc = construir_circuito(aplicar_oraculo, num_ancillas)
    qct = transpile(qc, basis_gates=BASIS, optimization_level=OPT)
    ops = qct.count_ops()
    return qc.num_qubits, ops.get('cx', 0), qct.depth(), sum(ops.values())
 
 
if __name__ == "__main__":
    print(f"Caso de salón: {g}^x = {h} (mod {p})   |   base {BASIS}, "
          f"optimization_level={OPT}, conectividad ideal (sin SWAPs)\n")
 
    filas = []
    for nombre, aplicar, nanc in ORACULOS:
        qubits, cx, prof, total = medir(aplicar, nanc)
        filas.append((nombre, qubits, cx, prof, total))
 
    # ---- tabla por consola ----
    print(f"{'Oráculo':<12} | {'Cúbits':>6} | {'CX':>7} | {'Profundidad':>11} | {'Total puertas':>13}")
    print("-" * 60)
    for nombre, qubits, cx, prof, total in filas:
        print(f"{nombre:<12} | {qubits:>6} | {cx:>7} | {prof:>11} | {total:>13}")
    print("-" * 60)
    print("\nEstas cifras son las de la Tabla 8.1 (comparación entre oráculos).")
    print("NO son las del mapeo a ibm_marrakesh (allí hay SWAPs y base 'cz').")

Caso de salón: 5^x = 19 (mod 23)   |   base ['cx', 'rz', 'sx', 'x'], optimization_level=1, conectividad ideal (sin SWAPs)

[Clásico] Factores calculados para Registro A (base 5): [5, 2, 4, 16, 3]
[Clásico] Factores calculados para Registro B (base 17): [17, 13, 8, 18, 2]
[Cuántico] Exponenciación por puertas: base g=5 (reg_A), base h^-1=17 (reg_B)
Oráculo      | Cúbits |      CX | Profundidad | Total puertas
------------------------------------------------------------
Matrices     |     15 |   10092 |       40258 |         68248
Beauregard   |     22 |   25726 |       34903 |         65607
Montgomery   |     26 |   20304 |       21293 |         47102
------------------------------------------------------------

Estas cifras son las de la Tabla 8.1 (comparación entre oráculos).
NO son las del mapeo a ibm_marrakesh (allí hay SWAPs y base 'cz').
